In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# Installs are quiet (-q) so the output stays readable on a projector.
!pip install -q google-genai chromadb rank-bm25 numpy

import os, json, time                    # standard library
from google import genai                 # the SDK
from google.genai import types           # config and content types
import numpy as np
import chromadb
from rank_bm25 import BM25Okapi

# ── Your API key ──────────────────────────────────────────────────────
# The key lives in Colab Secrets, never in the notebook. If this cell
# fails, that is almost always why.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    if not API_KEY:
        raise ValueError("empty")
except Exception:
    raise SystemExit(
        "\n" + "=" * 68 +
        "\nNo API key found.\n"
        "\n  1. Click the KEY icon in the left sidebar of Colab."
        "\n  2. Click 'Add new secret'."
        "\n  3. Name it exactly:  GEMINI_API_KEY"
        "\n  4. Paste your key from aistudio.google.com"
        "\n  5. Turn ON 'Notebook access' for this notebook."
        "\n  6. Run this cell again."
        "\n" + "=" * 68
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash-lite"          # fast and cheap; what we use all week
EMBED_MODEL = "gemini-embedding-001"    # free tier, which is what makes Day 2 possible

print("Ready. Model:", MODEL)

In [ ]:
# ── The document set ──────────────────────────────────────────────────
# 15 short policy documents plus one SCANNED page with no text layer.
# They live in the course repository so this works without any live
# service. Set RAW_BASE to your repository's raw URL.

RAW_BASE = "https://raw.githubusercontent.com/YOUR-ACCOUNT/YOUR-REPO/main/notebooks/data"

FILES = [
    "01_leave_policy_en.txt", "02_training_policy_en.txt",
    "03_data_governance_en.txt", "04_security_policy_en.txt",
    "05_procurement_en.txt", "06_remote_work_en.txt",
    "07_ai_ethics_en.txt", "08_records_en.txt",
    "09_leave_policy_ar.txt", "10_data_protection_ar.txt",
    "11_training_ar.txt", "12_security_ar.txt",
    "13_conduct_en.txt", "14_it_support_en.txt",
    "15_meeting_rooms_en.txt",
]

os.makedirs("data", exist_ok=True)
for name in FILES + ["scan_p03.png"]:
    if not os.path.exists("data/" + name):
        !wget -q -O data/{name} {RAW_BASE}/{name}

print("Downloaded:", len(os.listdir("data")), "files")
print(sorted(os.listdir("data"))[:5], "...")

In [ ]:
# Load and clean into a list of {text, source, page} dicts.
# PRE-WRITTEN. Note the metadata: without source and page you cannot cite,
# and adding it later means re-processing everything.

def load_documents(folder="data"):
    docs = []
    for name in sorted(os.listdir(folder)):
        if not name.endswith(".txt"):
            continue
        with open(os.path.join(folder, name), encoding="utf-8") as fh:
            raw = fh.read()

        # Cleaning: collapse runs of blank lines, strip trailing spaces.
        lines = [ln.rstrip() for ln in raw.split("\n")]
        cleaned = "\n".join(lines)
        while "\n\n\n" in cleaned:
            cleaned = cleaned.replace("\n\n\n", "\n\n")

        docs.append({"text": cleaned.strip(), "source": name, "page": 1})
    return docs


docs = load_documents()
print(len(docs), "documents loaded")
print()
print(docs[0]["text"][:300])

## Multimodal ingestion — the applied bit

One of the files you downloaded is `scan_p03.png`: a **photograph of a page**.
It has no text layer. Open it with any text tool and you get an empty string.

Half the Arabic PDFs in a government archive look like this.

So we make the vision model a **stage in the pipeline**: page image in, text
out, into the same list of dicts as everything else. Nothing downstream will
know the difference — and that is what makes it a pipeline stage rather than
a demo.

The instruction matters. Ask vaguely and you get a helpful summary that has
silently lost the detail you needed. Ask for a transcription and say
*"do not summarise"*.

In [ ]:
# Vision extraction. PRE-WRITTEN, and the highest-impact cell in the notebook.
import pathlib

page_bytes = pathlib.Path("data/scan_p03.png").read_bytes()

VISION_PROMPT = (
    "Transcribe this page exactly as written. "
    "Keep any Arabic text in Arabic. "
    "Keep table rows on separate lines. "
    "Do not summarise, do not explain, do not add commentary."
)

vresp = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=page_bytes, mime_type="image/png"),
        VISION_PROMPT,
    ],
)

print(vresp.text)

# Same shape as every other document — that is the whole point.
docs.append({"text": vresp.text, "source": "circular_2024_scan.pdf", "page": 3})
print()
print("Corpus is now", len(docs), "documents.")

In [ ]:
# TODO ─ The chunker. The structure is written; the loop body is blanked.
#
# Rules to implement:
#   * split on paragraphs FIRST (structure before size)
#   * start a new chunk when the current one would exceed size
#   * carry `overlap` tokens of the previous chunk into the next one

def chunk(text, size=500, overlap=50):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""

    for para in paras:
        # Roughly 4 characters per token, so size*4 characters ~ size tokens.
        if len(current) + len(para) < size * 4:
            current += para + "\n\n"
        else:
            # ← TODO (2 lines):
            #   1. append the finished chunk (stripped) to `chunks`
            #   2. start `current` again from the LAST overlap*4 characters
            #      of the finished chunk, then add this paragraph
            pass

    if current.strip():
        chunks.append(current.strip())
    return chunks


# Flatten every document into chunk records, keeping the metadata.
records = [
    {"text": ch, "source": d["source"], "page": d["page"]}
    for d in docs
    for ch in chunk(d["text"])
]

print(len(records), "chunks from", len(docs), "documents")

In [ ]:
# Inspect three chunks. Check nothing is cut mid-word and every chunk
# still makes sense on its own — a chunk is retrieved ALONE.
for r in records[:3]:
    print("─" * 70)
    print(r["source"], "· page", r["page"], "·", len(r["text"]), "chars")
    print(r["text"][:400])

# Arabic prints right-to-left in the output; that is the terminal doing its
# job, not your data being broken. Check len() rather than trusting your eye.
print("─" * 70)
arabic_chunks = [r for r in records if r["source"].endswith("_ar.txt")]
print("Arabic chunks:", len(arabic_chunks),
      "| avg chars:", sum(len(r["text"]) for r in arabic_chunks) // max(len(arabic_chunks), 1))

In [ ]:
# Embed every chunk. PRE-WRITTEN — including the batching and the sleep,
# which are there because the free tier WILL rate-limit you today.

def embed(texts, task="RETRIEVAL_DOCUMENT", batch=32, pause=1.0):
    """task is RETRIEVAL_DOCUMENT for chunks, RETRIEVAL_QUERY for questions.
    Using the wrong one quietly costs you accuracy."""
    out = []
    for i in range(0, len(texts), batch):
        r = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts[i:i + batch],
            config=types.EmbedContentConfig(task_type=task),
        )
        out += [e.values for e in r.embeddings]
        print(f"  embedded {min(i + batch, len(texts))}/{len(texts)}")
        time.sleep(pause)          # do not remove this line
    return out


vectors = embed([r["text"] for r in records])
print()
print(len(vectors), "vectors of", len(vectors[0]), "dimensions")

In [ ]:
# Store in Chroma with the metadata attached.
db = chromadb.Client()

# Guard: re-running this cell without deleting first is the second most
# common failure of the day (after the API key).
try:
    db.delete_collection("policies")
    print("Deleted the existing collection.")
except Exception:
    pass

col = db.create_collection("policies")

col.add(
    ids=[f"c{i}" for i in range(len(records))],
    documents=[r["text"] for r in records],
    embeddings=vectors,
    metadatas=[{"source": r["source"], "page": r["page"]} for r in records],
)

print("Stored", col.count(), "chunks.")

In [ ]:
# search(query, k) — embed the question, return the nearest chunks.
def search(query, k=4):
    qv = embed([query], task="RETRIEVAL_QUERY", pause=0)[0]
    res = col.query(query_embeddings=[qv], n_results=k)

    hits = []
    for text, meta in zip(res["documents"][0], res["metadatas"][0]):
        hits.append({"text": text, "source": meta["source"], "page": meta["page"]})
    return hits


for hit in search("How many leave days does grade 11 get?"):
    print("─" * 70)
    print(f"[{hit['source']} p.{hit['page']}]")
    print(hit["text"][:220])

In [ ]:
# answer(query) — retrieve, build a grounded prompt, generate WITH citations.
GROUNDED = """You are a policy assistant. Answer using ONLY the reference
material between the tags. Cite the source file and page for every fact,
like this: (leave_policy.txt p.1). If the material does not contain the
answer, say you do not know.

<reference>
{context}
</reference>

Question: {question}"""


def answer(query, k=4):
    hits = search(query, k)
    context = "\n\n".join(
        f"[{h['source']} p.{h['page']}]\n{h['text']}" for h in hits)

    r = client.models.generate_content(
        model=MODEL,
        contents=GROUNDED.format(context=context, question=query),
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return r.text


print(answer("How many annual leave days does grade 11 get, and can they be carried over?"))
print()
print("─" * 70)
print(answer("What is the capital of Brazil?"))    # should decline: not in the docs

In [ ]:
# TODO ─ Ask three questions about YOUR OWN domain.
#
# Use the corpus above for now; on Wednesday you point this at your own
# documents. Write questions a real colleague would actually type.

my_questions = [
    "TODO: your first question",       # ← TODO
    "TODO: your second question",      # ← TODO
    "TODO: your third question",       # ← TODO
]

for q in my_questions:
    print("=" * 70)
    print("Q:", q)
    print(answer(q))

# Note for each one: was the retrieved chunk the right chunk? If not, was it
# a chunking problem or a retrieval problem? That distinction is Part B.

# Part B — the ceiling

You have working retrieval. Now the part that earns the course its title.

## Where meaning-based search falls over

Embeddings encode **meaning**. So what happens when the query has no meaning
to encode?

`SDAIA-F-CRS-201-01-V1` is not a concept. It is a string of characters. Its
embedding lands somewhere close to arbitrary, near other document-ish text —
which is exactly not what you wanted.

And this is what a professional user types. They know which document they
want; they are giving you its number. Run the next cell and watch it miss.

In [ ]:
# Watch pure vector search fail on an exact identifier.
QUERY_ID = "SDAIA-F-CRS-201-01-V1"

print("Query:", QUERY_ID)
print()
for hit in search(QUERY_ID, k=4):
    contains = QUERY_ID in hit["text"]
    print(f"[{hit['source']}] contains the id: {contains}")
    print("  ", hit["text"][:120].replace("\n", " "))

print()
print("The document that IS this form is:", "02_training_policy_en.txt")

In [ ]:
# BM25 — keyword scoring over exactly the same chunks.
# Decades old, boring, and unbeatable at exact strings.
tokenised = [r["text"].lower().split() for r in records]
bm25 = BM25Okapi(tokenised)

scores = bm25.get_scores(QUERY_ID.lower().split())
top = np.argsort(scores)[::-1][:3]

for i in top:
    print(f"{scores[i]:6.2f}  {records[i]['source']}  "
          f"contains id: {QUERY_ID in records[i]['text']}")

In [ ]:
# TODO ─ hybrid_search(). Everything is written except the weighting line.

def norm(x):
    """Squash to 0..1 so two different score scales can be added."""
    x = np.array(x, dtype=float)
    spread = np.ptp(x)
    return (x - x.min()) / (spread + 1e-9)


# Cache the document vectors once so hybrid search is not slow.
DOC_MATRIX = np.array(vectors, dtype=float)


def vector_scores(query):
    qv = np.array(embed([query], task="RETRIEVAL_QUERY", pause=0)[0], dtype=float)
    # Cosine similarity against every chunk.
    dots = DOC_MATRIX @ qv
    return dots / (np.linalg.norm(DOC_MATRIX, axis=1) * np.linalg.norm(qv) + 1e-9)


def hybrid_search(query, k=4, alpha=0.5):
    kw = norm(bm25.get_scores(query.lower().split()))
    vec = norm(vector_scores(query))

    # ← TODO (1 line): combine the two score arrays using alpha.
    #   alpha = 1.0 should be pure vector search
    #   alpha = 0.0 should be pure keyword search
    score = None

    top = np.argsort(score)[::-1][:k]
    return [records[i] for i in top]

In [ ]:
# Re-run the query that failed. Watch it succeed.
print("Query:", QUERY_ID, "| hybrid search")
print()
for hit in hybrid_search(QUERY_ID, k=4):
    print(f"[{hit['source']}] contains the id: {QUERY_ID in hit['text']}")

print()
print("Same query, same chunks, same embeddings. One weighted average.")

## Re-ranking — retrieve 20 cheap, keep the best 4

Retrieval is fast and shallow. Judging relevance properly is slow and
expensive. So do both, in that order:

1. Hybrid search returns **20** candidates. Milliseconds, near-zero cost.
2. Something more expensive scores each one for relevance, 0 to 10.
3. Keep the best **4**. Only those go in the prompt.

Your recall comes from the cheap wide net; your precision comes from the
expensive judge; and the prompt stays small, so generation stays cheap.

Yes — this is the "k = 20" I warned you about yesterday. The difference is
that sixteen of them never reach the model.

In [ ]:
# LLM-as-reranker. Slower than a proper cross-encoder, but needs nothing
# extra installed and the idea is identical.

RERANK_SCHEMA = {
    "type": "object",
    "properties": {"scores": {"type": "array", "items": {"type": "integer"}}},
    "required": ["scores"],
}


def rerank(query, candidates, keep=4):
    listing = "\n\n".join(
        f"[{i}] {c['text'][:400]}" for i, c in enumerate(candidates))

    r = client.models.generate_content(
        model=MODEL,
        contents=(f"Question: {query}\n\nScore each passage 0-10 for how well "
                  f"it answers that question. Return one score per passage, in "
                  f"order.\n\n{listing}"),
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RERANK_SCHEMA,
            temperature=0.0,
        ),
    )

    scores = json.loads(r.text)["scores"]
    ranked = sorted(zip(scores, candidates), key=lambda p: p[0], reverse=True)
    return [c for _, c in ranked[:keep]]


def reranked_search(query, k=4):
    wide = hybrid_search(query, k=20)      # cheap and wide
    return rerank(query, wide, keep=k)     # expensive and narrow


for hit in reranked_search("Can I carry unused leave into next year?"):
    print("─" * 70)
    print(f"[{hit['source']}]", hit["text"][:180].replace("\n", " "))

In [ ]:
# Query rewriting — the user's question is rarely the best search string.
def rewrite(question, history=""):
    r = client.models.generate_content(
        model=MODEL,
        contents=(f"Conversation so far:\n{history}\n\n"
                  f"User's latest message: {question}\n\n"
                  "Rewrite it as a standalone search query. Keep any "
                  "identifiers or numbers exactly. Return only the query."),
        config=types.GenerateContentConfig(temperature=0.0),
    )
    return r.text.strip()


history = "User: How many annual leave days does grade 11 get?\nAssistant: 30 working days."
print("Raw follow-up :", "and what about carrying it over?")
print("Rewritten     :", rewrite("and what about carrying it over?", history))

## The golden set — this is the part almost nobody does

Ten questions **you already know the answers to**, written against your own
documents, **before** you tune anything. For each one, record a phrase that
must appear in whatever comes back.

Then: what fraction did retrieval get right? That is your hit rate, and it is
the difference between *"it looked right"* and *"my retrieval is 9 out of 10"*.

Write good ones:

* questions a real user would type, not questions your system will pass
* include the awkward ones — an identifier, an acronym, an Arabic question
* include **one whose answer is not in the corpus** ("I don't know" is correct)
* write them before you tune, or you are marking your own homework

In [ ]:
# TODO ─ Fill in the golden set. Three are written as examples;
#        add at least five more of your own.

GOLDEN = [
    {"q": "How many annual leave days does grade 11 get?",
     "must_contain": "30 working days"},
    {"q": "What is SDAIA-F-CRS-201-01-V1?",
     "must_contain": "SDAIA-F-CRS-201-01-V1"},
    {"q": "How quickly must a data breach be reported?",
     "must_contain": "twenty-four hours"},

    # ← TODO (5+ entries): add your own, in the same shape.
    # Include one identifier question, one Arabic question, and one whose
    # answer is NOT in the corpus at all.
]

print(len(GOLDEN), "golden questions")

In [ ]:
# evaluate(search_fn) → hit rate. PRE-WRITTEN.
def evaluate(search_fn, k=4, verbose=False):
    hits = 0
    for item in GOLDEN:
        got = " ".join(r["text"] for r in search_fn(item["q"], k))
        ok = item["must_contain"].lower() in got.lower()
        hits += ok
        if verbose:
            print(("  PASS  " if ok else "  MISS  ") + item["q"][:60])
    return hits / len(GOLDEN)


print("Naive vector search, question by question:")
evaluate(search, verbose=True)

In [ ]:
# Score all three retrievers and print the comparison table.
# This table is what goes in your Thursday presentation.
results = []
for name, fn in [("naive vector", search),
                 ("hybrid", hybrid_search),
                 ("hybrid + rerank", reranked_search)]:
    rate = evaluate(fn)
    results.append((name, rate))

print()
print(f"{'retriever':<20} {'hit rate':>10}")
print("-" * 32)
for name, rate in results:
    print(f"{name:<20} {rate * len(GOLDEN):>5.0f}/{len(GOLDEN)}  {rate:>5.0%}")

## Reflection

Fill these in before you close the notebook. This is what I check when I come round.

**Which retriever won, and by how much?**

> _your answer here_

**Which golden questions still fail with the best retriever? What do they have in common?**

> _your answer here_

**Is the remaining failure a chunking problem, a retrieval problem, or a question that genuinely cannot be answered from this corpus?**

> _your answer here_

**What would you change first if you had another hour?**

> _your answer here_

## If this breaks

The three most likely failures, and what to do about each.

| Symptom | Cause | Fix |
|---|---|---|
| `429 RESOURCE_EXHAUSTED` while embedding | Free-tier rate limit — everyone in the room is embedding at once | The batching and `time.sleep` in the embed cell handle this. Raise `pause` to 2.0 if it persists |
| `Collection policies already exists` | You re-ran the storage cell | The `delete_collection` guard is already in that cell — run the whole cell, not just part of it |
| Arabic looks scrambled when printed | Terminal bidi rendering, not your data | Check `len(text)` and search for a substring instead of trusting the visual order |